# Sales Data Cleaning Project

## Project Overview

This project cleans a badly structured Excel sales file and converts it into a tidy dataset ready for analysis and SQL querying.

The original file contains multi-row headers and sales values spread across multiple columns. The goal is to restructure the data into a clean long-format table with four columns: `order_date`, `segment`, `ship_mode`, and `sales`.

## Tools Used

- Python
- pandas
- Jupyter Notebook
- MySQL
- MySQL Workbench

## Final Output

The cleaned dataset will be exported as an Excel/CSV file and loaded into MySQL for SQL analysis.


In [ ]:
import pandas as pd
from pathlib import Path
from getpass import getpass
from sqlalchemy import create_engine

In [ ]:
project_folder=Path.cwd()
raw_folder = project_folder / "raw_data"
cleaned_folder = project_folder / "clean_data"
outputs_folder = project_folder / "output"
sql_folder = project_folder / "sql"
notes_folder = project_folder / "notes"

In [ ]:
def audit_df(df, name):
    print(f"Audit: {name}")
    print("Shape:", df.shape)
    print("Missing values:")
    print(df.isna().sum())
    print("Duplicate rows:", df.duplicated().sum())
    display(df.head())

In [ ]:
raw_file=raw_folder/"2BadlyStructuredSalesData22-220823-192255.xlsx"
df_raw=pd.read_excel(raw_file,header=None)

In [ ]:
audit_df(df_raw, "raw data after loading")

In [ ]:
df=df_raw.copy()

In [ ]:
ship_mode=df.iloc[0].ffill()
segment=df.iloc[1]

In [ ]:
new_columns = []

for i in range(len(df.columns)):
    if i == 0:
        new_columns.append("order_date")
    else:
        new_columns.append(f"{ship_mode[i]}_{segment[i]}")

In [ ]:
df.columns = new_columns
df_wide = df.iloc[3:].reset_index(drop=True)

In [ ]:
audit_df(df_wide, "raw data with new column names")

In [ ]:
df_long = df_wide.melt(
    id_vars="order_date",
    var_name="shipmode_segment",
    value_name="sales"
)


In [ ]:
audit_df(df_long, "raw data long format")

In [ ]:
df_long = df_long.dropna(subset=["sales"]).reset_index(drop=True)

In [ ]:
audit_df(df_long, "cleaned data long format")

In [ ]:
df_long[["ship_mode", "segment"]] = df_long["shipmode_segment"].str.split("_", expand=True)
df_long = df_long.drop(columns="shipmode_segment")

In [ ]:
df_long["order_date"] = pd.to_datetime(df_long["order_date"], errors="coerce")


In [ ]:
df_long["sales"] = pd.to_numeric(df_long["sales"], errors="coerce")


In [ ]:
audit_df(df_long, "cleaned data long format")

In [ ]:
df_clean = df_long[["order_date", "segment", "ship_mode", "sales"]]

In [ ]:
audit_df(df_clean, "final cleaned data before export")

df_clean["sales"].describe()


## Final Data Validation

The final cleaned dataset contains 822 rows and 4 columns.

There are no missing values in the key columns: `order_date`, `segment`, `ship_mode`, and `sales`.

There are no duplicate rows.

The `sales` column contains only positive values. The minimum sales value is 1.167 and the maximum sales value is 23661.228.

The median sales value is 159.975, while the mean is 476.547. This suggests the sales data is right-skewed, most orders are smaller, while a few large orders increase the average.

Based on these checks, the cleaned dataset is ready for export and SQL analysis.

In [ ]:
cleaned_excel_file = cleaned_folder/"cleaned_sales_data.xlsx"
cleaned_csv_file = cleaned_folder/"cleaned_sales_data.csv"
df_clean.to_excel(cleaned_excel_file, index=False)
df_clean.to_csv(cleaned_csv_file, index=False)

## Required Packages

This notebook uses the following Python packages:

- pandas
- openpyxl
- SQLAlchemy
- PyMySQL

If needed, install them with:

`%pip install pandas openpyxl sqlalchemy pymysql`



In [ ]:
mysql_user = "root"
mysql_password = getpass("Enter MySQL password: ")
mysql_host = "localhost"
mysql_port = 3306
mysql_database = "badly_structured_sales_analysis"

engine = create_engine(
    f"mysql+pymysql://{mysql_user}:{mysql_password}@{mysql_host}:{mysql_port}/{mysql_database}"
)


In [ ]:
df_clean.to_sql(
    name="sales_clean",
    con=engine,
    if_exists="replace",
    index=False
)

In [ ]:
pd.read_sql("SELECT COUNT(*) AS total_rows FROM sales_clean", engine)


The SQL row count matched the final cleaned dataset row count, confirming that all 822 cleaned rows were uploaded successfully.
